# Projeto 2 — Sprint 2 | Testes de hipótese

**Tema:** tempo de carregamento de caminhões com grãos  
**Fonte:** Planilha 1 (antes; n=300) e Planilha 2 (depois; n=100).

Este notebook foi preparado para execução no Google Colab. Quando solicitado, envie a planilha indicada pelo seletor de arquivos. Os dados não são alterados.

**Hipótese direcional do projeto:** a intervenção busca reduzir o tempo médio de carregamento e aumentar a proporção de carregamentos dentro do intervalo de referência. Os testes são unilaterais à esquerda (média) e à direita (proporção).


## Questão 1 — Contexto e proporções observadas

A Sprint 2 determina que p₀ seja a probabilidade de uma observação estar no intervalo [a,b] escolhido na Questão 5 da Sprint 1, e que p̂ seja calculado na Planilha 2 usando o mesmo intervalo. A Questão 5 adotou [20,43] minutos.

### 1.1 Carregamento das planilhas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files
from io import BytesIO

COLUNA = "Tempo de carregamento (min)"

print("Envie a Planilha 1 (antes da intervenção).")
up1 = files.upload()
arquivo1 = next(iter(up1))
df_antes = pd.read_excel(BytesIO(up1[arquivo1]))

print("Envie a Planilha 2 (depois da intervenção).")
up2 = files.upload()
arquivo2 = next(iter(up2))
df_depois = pd.read_excel(BytesIO(up2[arquivo2]))

for nome, df in [("Planilha 1", df_antes), ("Planilha 2", df_depois)]:
    if COLUNA not in df.columns:
        raise ValueError(f"{nome}: coluna '{COLUNA}' não encontrada. Colunas: {df.columns.tolist()}")

antes = pd.to_numeric(df_antes[COLUNA], errors="coerce").dropna()
depois = pd.to_numeric(df_depois[COLUNA], errors="coerce").dropna()
print(f"n antes = {len(antes)} | n depois = {len(depois)}")


## Atenção à consistência do intervalo

Na Sprint 1, a Questão 5 calculou p₀ para **[20,43]** minutos. Porém, a Questão 7.4(b) apresentou p̂ = 57/100 = 57% usando **[20,30]** minutos. Esses valores não representam o mesmo evento e não podem ser comparados diretamente no teste da proporção. Para atender à orientação explícita da Sprint 2, este notebook mantém [20,43] e recalcula p̂ da Planilha 2 nesse mesmo intervalo. O resultado original em [20,30] é mostrado apenas como referência da Sprint 1.

In [ ]:
a = 20
b = 43

if a > b:
    raise ValueError("O limite inferior a deve ser menor ou igual ao limite superior b.")

n1, n2 = len(antes), len(depois)
media_antes = antes.mean()
dp_antes = antes.std(ddof=1)

# p0: probabilidade do modelo normal ajustado à Planilha 1, como na Questão 5 da Sprint 1.
p0 = stats.norm.cdf(b, loc=media_antes, scale=dp_antes) - stats.norm.cdf(a, loc=media_antes, scale=dp_antes)

# p-hat corrigido: proporção observada na Planilha 2 dentro do MESMO intervalo [20,43].
sucessos = int(depois.between(a, b, inclusive="both").sum())
phat = sucessos / n2

# Valor publicado na Questão 7.4(b) da Sprint 1, calculado em intervalo diferente [20,30].
sucessos_20_30 = int(depois.between(20, 30, inclusive="both").sum())
phat_20_30 = sucessos_20_30 / n2

print(f"Intervalo comum para o teste: [{a}, {b}] minutos")
print(f"p0 (modelo normal da Planilha 1) = {p0:.6%}")
print(f"p-hat corrigido (Planilha 2, mesmo intervalo) = {sucessos}/{n2} = {phat:.6%}")
print(f"Referência Sprint 1 em [20,30] = {sucessos_20_30}/{n2} = {phat_20_30:.2%}")


### Verificação adicional: teste binomial exato

Como p₀ está muito próximo de 1, a quantidade esperada de fracassos sob H₀ é n(1−p₀), que pode ser menor que 5. Nessa situação, a aproximação normal do teste z pode não ser adequada. O teste z e seus gráficos são apresentados por exigência do roteiro da aula; abaixo, incluímos o teste binomial exato como verificação complementar.

In [ ]:
fracassos_esperados = n2 * (1 - p0)
p_valor_binomial = stats.binom.sf(sucessos - 1, n2, p0)  # H1: p > p0
print(f'Fracassos esperados sob H0 = {fracassos_esperados:.4f}')
print(f'p-valor do teste binomial exato unilateral à direita = {p_valor_binomial:.8g}')
for alpha in (0.05, 0.01):
    print(f'α={alpha:.0%}: ' + ('REJEITAR H0' if p_valor_binomial < alpha else 'NÃO REJEITAR H0'))


## Questão 3a — Formulação das hipóteses

Como o objetivo é verificar se aumentou a proporção de carregamentos na faixa de referência [20,43], o teste é unilateral à direita:

- **H₀:** p = p₀
- **H₁:** p > p₀

O valor p₀ é a probabilidade calculada na Questão 5 da Sprint 1; p̂ é a proporção amostral da Planilha 2, recalculada no mesmo intervalo.

In [ ]:
# Estatística z para uma proporção, conforme a fórmula de teste apresentada na Aula 2.
erro_padrao = np.sqrt(p0 * (1 - p0) / n2)
z_calculado = (phat - p0) / erro_padrao
p_valor = 1 - stats.norm.cdf(z_calculado)

print(f"p0 = {p0:.8f}")
print(f"p-hat = {phat:.8f}")
print(f"n = {n2}")
print(f"Erro-padrão sob H0 = {erro_padrao:.8f}")
print(f"z calculado = {z_calculado:.6f}")
print(f"p-valor unilateral à direita = {p_valor:.6g}")


## Questão 3b — Teste para α = 5% e α = 1%

Para um teste unilateral à direita, a região de rejeição é z ≥ z crítico. A região de não rejeição é z < z crítico.

In [ ]:
def resultado_teste_proporcao(alpha):
    z_critico = stats.norm.ppf(1 - alpha)  # cauda direita
    rejeita = z_calculado >= z_critico

    print(f"\nNível de significância α = {alpha:.0%}")
    print(f"Valor crítico: {z_critico:.4f}")
    print(f"Região de rejeição: z ≥ {z_critico:.4f}")
    print(f"Região de não rejeição: z < {z_critico:.4f}")
    print(f"z calculado: {z_calculado:.4f}")
    print(f"p-valor unilateral: {p_valor:.6g}")
    print("Decisão: " + ("REJEITAR H0" if rejeita else "NÃO REJEITAR H0"))
    if rejeita:
        print("Conclusão: há evidência estatística, ao nível adotado, de que a proporção de carregamentos no intervalo [20,43] aumentou em relação a p0.")
    else:
        print("Conclusão: não há evidência estatística suficiente, ao nível adotado, para afirmar que a proporção no intervalo [20,43] aumentou em relação a p0.")
    return z_critico, rejeita

res_prop_5 = resultado_teste_proporcao(0.05)
res_prop_1 = resultado_teste_proporcao(0.01)


In [ ]:
def grafico_regioes_proporcao(alpha):
    zcrit = stats.norm.ppf(1 - alpha)
    x = np.linspace(-4, 4, 1000)
    y = stats.norm.pdf(x)

    plt.figure(figsize=(9, 4.8))
    plt.plot(x, y, label="Distribuição normal padrão")
    plt.fill_between(x, 0, y, where=(x >= zcrit), alpha=0.30,
                     label="Região de rejeição de H0")
    plt.fill_between(x, 0, y, where=(x < zcrit), alpha=0.12,
                     label="Região de não rejeição de H0")
    plt.axvline(zcrit, linestyle="--", label=f"z crítico = {zcrit:.3f}")
    plt.axvline(z_calculado, linestyle="-.", label=f"z calculado = {z_calculado:.3f}")
    plt.title(f"Teste unilateral à direita da proporção — α = {alpha:.0%}")
    plt.xlabel("z")
    plt.ylabel("Densidade")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

grafico_regioes_proporcao(0.05)
grafico_regioes_proporcao(0.01)


## Questão 3c — Comparação com a análise preliminar

A Sprint 1 reportou 57% na faixa [20,30], mas essa faixa difere da faixa [20,43] usada para p₀ na Questão 5. Por isso, a comparação original não é válida para inferência sobre uma mesma proporção. Neste notebook, p̂ foi recalculado em [20,43], e a conclusão deve ser comparada à análise descritiva usando esse intervalo consistente. Não rejeitar H₀ não significa provar que as proporções são iguais.

## Observações metodológicas

1. O intervalo [20,43] foi escolhido pelo grupo na Sprint 1; o relatório recomenda justificar sua relevância operacional.
2. A aproximação normal do teste z para proporção deve ser avaliada pelas condições de sucesso e fracasso esperados sob H₀. Como p₀ pode estar muito próximo de 1, verifique n·p₀ e n·(1−p₀); se a condição não for atendida, a aproximação z pode ser inadequada. Consulte o docente sobre eventual teste binomial exato.
3. A análise é observacional antes/depois e não prova causalidade.